# PST-Parser — generazione downstream su Colab

Lanciatore della generazione di codice (Qwen2.5-Coder-7B-Instruct, bf16, backend `hf`): **nessuna logica propria**. Ogni cella chiama gli script e i comandi del progetto e si ferma con un errore se un controllo fallisce.

Una run è legata alla tripla (archivio sorgente, freeze o record di revisione, revisione del modello), registrata in `run-identity.json` su Drive alla prima esecuzione e confrontata a ogni ripresa.

Che cosa serve:
1. l'archivio sorgente creato in locale con `git -c core.autocrlf=false archive --format=zip -o pst-src.zip HEAD` e il tree stampato da `git rev-parse "HEAD^{tree}"`;
2. un archivio `inputs.zip` con una cartella per journal; ogni cartella contiene `variants.jsonl` (da `render`, con `--envelope` per il nucleo BigCodeBench), `generate.json` (configurazione v2, con `"gpu": "NVIDIA L4"`: `generate` rifiuta ogni altra GPU) e il record di revisione nominato da `review_record`;
3. per le run `final`, lo SHA-256 di `protocol/protocol-freeze.json`.

Regole: una RUN_ID nuova per ogni run nuova; non modificare mai i file di una RUN_ID esistente; la ripresa usa la stessa identità e riprende il journal da solo.

In [ ]:
import subprocess

subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    check=True,
)

In [ ]:
import hashlib
import os
import re
import sys
import zipfile
from pathlib import Path

from google.colab import files

EXPECTED_TREE = "INSERIRE_TREE_40_CARATTERI"  # git rev-parse "HEAD^{tree}" in locale
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_TREE):
    raise ValueError("Impostare il tree del commit da eseguire")

uploaded = files.upload()  # selezionare solo pst-src.zip
if len(uploaded) != 1:
    raise ValueError("Selezionare un solo archivio sorgente ZIP")
archive_name, archive_bytes = next(iter(uploaded.items()))
SOURCE_ARCHIVE = Path("/content") / archive_name
SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
PROJECT = Path("/content/pst-src")
if PROJECT.exists():
    raise FileExistsError("Sorgente gia estratta: usare un runtime nuovo")
PROJECT.mkdir()
with zipfile.ZipFile(SOURCE_ARCHIVE) as bundle:
    for member in bundle.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError("Archivio con un percorso esterno alla cartella sorgente")
        if Path(member.filename).name == ".env":
            raise ValueError("Rimuovere .env dall'archivio sorgente")
    bundle.extractall(PROJECT)
if not (PROJECT / "pyproject.toml").is_file():
    raise ValueError("L'archivio deve avere il progetto alla radice (git archive HEAD)")
# Fine riga LF, hash dei file chiave e tree ricalcolato dal contenuto.
subprocess.run(
    [
        sys.executable,
        str(PROJECT / "scripts" / "verify_archive.py"),
        str(SOURCE_ARCHIVE),
        "--expected-tree",
        EXPECTED_TREE,
    ],
    check=True,
)
os.chdir(PROJECT)
print("source archive sha256:", SOURCE_ARCHIVE_SHA256)

In [ ]:
# Solo gli extra del generatore (cu128 e train): il parser ha una sezione a parte.
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.15"], check=True)
EXTRAS = ["--extra", "cu128", "--extra", "train"]
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11", *EXTRAS], check=True)
UV_RUN = ["uv", "run", "--frozen", *EXTRAS]
EMPIRICAL = [*UV_RUN, "python", "-m", "pstparser.empirical"]
subprocess.run([*UV_RUN, "python", "-c", "import pstparser, torch, transformers"], check=True)

In [ ]:
import json

from google.colab import drive

drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf-cache"

RUN_ID = "ds-pilot-01"  # ds-<fase>-NN; cambiare per ogni run NUOVA
MODEL_REVISION = "c03e6d358207e414f1eca0bb1891e29f1db0e242"
PURPOSE = "pilot"  # "pilot" oppure "final"
FREEZE_SHA256 = None  # obbligatorio per "final": SHA-256 di protocol/protocol-freeze.json

if not re.fullmatch(r"ds-[a-z0-9]+-[0-9]{2}", RUN_ID):
    raise ValueError("RUN_ID deve avere la forma ds-<fase>-NN")
if not re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION):
    raise ValueError("Fissare la revisione del modello (40 caratteri)")
if PURPOSE not in {"pilot", "final"}:
    raise ValueError("PURPOSE deve essere pilot o final")
if (PURPOSE == "final") != (FREEZE_SHA256 is not None):
    raise ValueError("Una run final richiede FREEZE_SHA256; una run pilot non lo usa")

RUN_ROOT = Path("/content/drive/MyDrive/pst-downstream-runs") / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
IDENTITY = RUN_ROOT / "run-identity.json"
INPUTS_ARCHIVE = RUN_ROOT / "inputs.zip"
INPUTS = RUN_ROOT / "inputs"

if not INPUTS_ARCHIVE.exists():
    if IDENTITY.exists():
        raise FileNotFoundError("Identita registrata ma inputs.zip assente: non riprendere")
    uploaded = files.upload()  # selezionare solo inputs.zip
    if len(uploaded) != 1:
        raise ValueError("Selezionare un solo archivio inputs.zip")
    INPUTS_ARCHIVE.write_bytes(next(iter(uploaded.values())))
INPUTS_SHA256 = hashlib.sha256(INPUTS_ARCHIVE.read_bytes()).hexdigest()

identity = {
    "run_id": RUN_ID,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "source_tree": EXPECTED_TREE,
    "freeze_sha256": FREEZE_SHA256,
    "model_revision": MODEL_REVISION,
    "purpose": PURPOSE,
    "inputs_sha256": INPUTS_SHA256,
}
if IDENTITY.exists():
    if json.loads(IDENTITY.read_text(encoding="utf-8")) != identity:
        raise ValueError("RUN_ID gia associata a un'altra identita: usare una RUN_ID nuova")
    print("Ripresa della run", RUN_ID)
else:
    if INPUTS.exists():
        raise FileExistsError("Cartella inputs senza identita: usare una RUN_ID nuova")
    with zipfile.ZipFile(INPUTS_ARCHIVE) as bundle:
        for member in bundle.infolist():
            target = (INPUTS / member.filename).resolve()
            if not target.is_relative_to(INPUTS.resolve()):
                raise ValueError("inputs.zip con un percorso esterno alla cartella")
        bundle.extractall(INPUTS)
    IDENTITY.write_text(json.dumps(identity, indent=2) + "\n", encoding="utf-8")

JOURNALS = sorted(path.name for path in INPUTS.iterdir() if path.is_dir())
if not JOURNALS:
    raise ValueError("inputs.zip non contiene cartelle di journal")
for name in JOURNALS:
    if not re.fullmatch(r"[a-z0-9-]+", name):
        raise ValueError(f"Nome di journal non ammesso: {name}")
    configuration = json.loads((INPUTS / name / "generate.json").read_text(encoding="utf-8"))
    if configuration["model"]["revision"] != MODEL_REVISION:
        raise ValueError(f"{name}: revisione del modello diversa da MODEL_REVISION")
    if configuration["purpose"] != PURPOSE:
        raise ValueError(f"{name}: purpose diverso da PURPOSE")
    if not (INPUTS / name / "variants.jsonl").is_file():
        raise FileNotFoundError(f"{name}: manca variants.jsonl")
print("journal:", JOURNALS)

In [ ]:
if PURPOSE == "final":
    # Comando freeze-verify del progetto: il freeze deve combaciare con la sorgente.
    FREEZE = PROJECT / "protocol" / "protocol-freeze.json"
    if hashlib.sha256(FREEZE.read_bytes()).hexdigest() != FREEZE_SHA256:
        raise ValueError("protocol-freeze.json diverso da FREEZE_SHA256")
    subprocess.run([*EMPIRICAL, "freeze-verify", str(FREEZE)], check=True)
else:
    print("Run pilot: nessun freeze; il record di revisione entra nel manifest di ogni journal")

## Generazione

Un journal per cartella di `inputs.zip`, tutti con le stesse condizioni dichiarate nella loro configurazione (per esempio {T, H-gold, E} sul nucleo BigCodeBench). Il comando congela richieste, configurazione e sorgente nel manifest e scrive ogni risposta appena prodotta.

**Ripresa.** Dopo un'interruzione: runtime nuovo, stesso `pst-src.zip`, stesse impostazioni della cella dell'identita, poi rieseguire tutte le celle. Il comando rigenera solo le richieste mancanti e rifiuta la ripresa se richieste, configurazione, sorgente o ambiente sono cambiati. Una riga troncata nel journal ferma la run: non si cancella e non si rigenera, si annota e si usa una RUN_ID nuova per le richieste mancanti.

In [ ]:
for name in JOURNALS:
    source = INPUTS / name
    configuration = source / "generate.json"
    if PURPOSE == "final":
        # The final configuration is the repository copy with the same sha256: its
        # review_record reaches protocol/protocol-freeze.json of this repository.
        digest = hashlib.sha256(configuration.read_bytes()).hexdigest()
        matches = [
            path
            for path in sorted((PROJECT / "protocol" / "configs" / "final").glob("*.json"))
            if hashlib.sha256(path.read_bytes()).hexdigest() == digest
        ]
        if len(matches) != 1:
            raise ValueError(f"{name}: generate.json is not a final configuration of the archive")
        configuration = matches[0]
    subprocess.run(
        [
            *EMPIRICAL,
            "generate",
            str(source / "variants.jsonl"),
            str(configuration),
            str(RUN_ROOT / "journals" / name),
        ],
        check=True,
    )

In [ ]:
# Archivio di trasporto: journal, manifest, varianti, configurazioni, identita e freeze.
number = 1
while (RUN_ROOT / f"{RUN_ID}-transport-{number:02d}.zip").exists():
    number += 1
TRANSPORT = RUN_ROOT / f"{RUN_ID}-transport-{number:02d}.zip"
with zipfile.ZipFile(TRANSPORT, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(IDENTITY, "run-identity.json")
    bundle.write(INPUTS_ARCHIVE, "inputs.zip")
    for path in sorted((RUN_ROOT / "journals").rglob("*")):
        if path.is_file():
            bundle.write(path, f"journals/{path.relative_to(RUN_ROOT / 'journals').as_posix()}")
    if PURPOSE == "final":
        bundle.write(PROJECT / "protocol" / "protocol-freeze.json", "protocol-freeze.json")
print("archivio:", TRANSPORT)
print("sha256:", hashlib.sha256(TRANSPORT.read_bytes()).hexdigest())

## Parser per H-auto (fase successiva)

Il parser di H-auto ha un notebook a parte, `notebooks/hauto-parser.ipynb`: backend `hf`, revisione e adapter verificati da `parse-prompts --purpose pilot|final`. Va eseguito in un runtime distinto da quello della generazione, prima di ogni journal H-auto.